# Phase 4: Polymarket Forensics

Analyze prediction market data for insider trading signals around Trump Iran posts. Examines market categories, volume around key events, and computes suspicion scores based on timing proximity to posts, volume concentration, and market type.

## Step 1: Load Polymarket and Supporting Data

Load Polymarket Iran-related markets, the master timeline, and cleaned Iran posts.

In [1]:
import json
import pandas as pd
import numpy as np
import os
from datetime import datetime

PROCESSED = '../data/processed'
RAW = '../data/raw'

In [2]:
pm_markets = pd.read_csv(os.path.join(RAW, 'polymarket_markets.csv'))
pm_markets['volume'] = pd.to_numeric(pm_markets['volume'], errors='coerce').fillna(0)
pm_markets['created_at'] = pd.to_datetime(pm_markets['created_at'], errors='coerce', utc=True).dt.tz_localize(None)

print(f"Markets loaded: {len(pm_markets)}")
print(f"Total volume: ${pm_markets['volume'].sum():,.0f}")

# Load master for timing analysis
master = pd.read_csv(os.path.join(PROCESSED, 'master.csv'))
master['date'] = pd.to_datetime(master['date'])

iran_posts = pd.read_csv(os.path.join(PROCESSED, 'iran_posts_cleaned.csv'))
iran_posts['timestamp'] = pd.to_datetime(iran_posts['timestamp'])

Markets loaded: 23
Total volume: $30,048,626


## Step 2: Categorize Iran Markets

Classify each Polymarket question as military, diplomatic, or other based on keyword matching.

In [3]:
military_kw = ['strike', 'military', 'attack', 'war', 'invade', 'forces', 'bomb', 'nuke']
diplomatic_kw = ['deal', 'ceasefire', 'talk', 'negotiate', 'peace']

def categorize_market(question):
    q = question.lower()
    if any(kw in q for kw in military_kw):
        return 'military'
    elif any(kw in q for kw in diplomatic_kw):
        return 'diplomatic'
    return 'other'

pm_markets['category'] = pm_markets['question'].apply(categorize_market)

print("Market categories:")
for cat, group in pm_markets.groupby('category'):
    print(f"  {cat}: {len(group)} markets, ${group['volume'].sum():,.0f} volume")

Market categories:
  diplomatic: 3 markets, $3,935,503 volume
  military: 14 markets, $17,107,686 volume
  other: 6 markets, $9,005,437 volume


## Step 3: Volume Analysis Around Key Dates

Examine which markets were active on critical event dates and their associated volumes and post counts.

In [4]:
key_events = {
    '2026-02-28': {'event': 'Iran strikes begin', 'note': '$1.2M Polymarket profits from 6 new wallets'},
    '2026-03-09': {'event': '$38 intraday oil swing', 'note': 'Largest crude range in history'},
    '2026-03-23': {'event': 'Productive conversations fabrication', 'note': 'Iran denied talks'},
}

for date_str, info in key_events.items():
    date = pd.Timestamp(date_str)

    # Find markets that were open during this date
    end_dates = pd.to_datetime(pm_markets['end_date'], errors='coerce', utc=True).dt.tz_localize(None)
    active = pm_markets[
        (pm_markets['created_at'] <= date) &
        ((end_dates.isna()) | (end_dates >= date))
    ]

    # Find posts on this day
    day_posts = iran_posts[iran_posts['timestamp'].dt.date == date.date()]

    print(f"\n{date_str}: {info['event']}")
    print(f"  Active Iran markets: {len(active)}")
    print(f"  Combined volume: ${active['volume'].sum():,.0f}")
    print(f"  Iran posts that day: {len(day_posts)}")
    print(f"  Note: {info['note']}")
    if not active.empty:
        top = active.nlargest(3, 'volume')
        for _, m in top.iterrows():
            print(f"    ${m['volume']:>12,.0f} | {m['question'][:55]}")


2026-02-28: Iran strikes begin
  Active Iran markets: 0
  Combined volume: $0
  Iran posts that day: 3
  Note: $1.2M Polymarket profits from 6 new wallets

2026-03-09: $38 intraday oil swing
  Active Iran markets: 0
  Combined volume: $0
  Iran posts that day: 6
  Note: Largest crude range in history

2026-03-23: Productive conversations fabrication
  Active Iran markets: 0
  Combined volume: $0
  Iran posts that day: 5
  Note: Iran denied talks


## Step 4: Compute Insider Trading Suspicion Scores

For each market, calculate a suspicion score based on:
- Timing proximity to the nearest Trump post (30% weight)
- Volume concentration relative to total market (40% weight)
- Market category -- military markets score higher (30% weight)

In [5]:
suspicion_data = []

for _, market in pm_markets.iterrows():
    created = market['created_at']
    volume = market['volume']
    question = market['question']

    if pd.isna(created) or volume == 0:
        continue

    # Check if market was created close to a Trump post
    time_to_nearest_post = None
    for _, post in iran_posts.iterrows():
        diff = abs((post['timestamp'] - created).total_seconds()) / 3600  # hours
        if time_to_nearest_post is None or diff < time_to_nearest_post:
            time_to_nearest_post = diff

    # Volume concentration score (how much of total volume this market has)
    vol_pct = volume / pm_markets['volume'].sum() * 100

    # Calculate suspicion score
    timing_score = max(0, 100 - (time_to_nearest_post or 1000))
    volume_score = min(100, vol_pct * 10)
    category_score = 80 if market['category'] == 'military' else 40

    suspicion_score = (timing_score * 0.3 + volume_score * 0.4 + category_score * 0.3)

    suspicion_data.append({
        'question': question[:60],
        'slug': market['slug'],
        'volume': volume,
        'category': market['category'],
        'created_at': str(created),
        'hours_to_nearest_post': round(time_to_nearest_post, 1) if time_to_nearest_post else None,
        'volume_pct': round(vol_pct, 2),
        'suspicion_score': round(suspicion_score, 1),
    })

suspicion_df = pd.DataFrame(suspicion_data).sort_values('suspicion_score', ascending=False)

print("Top 10 most suspicious markets:")
for _, row in suspicion_df.head(10).iterrows():
    print(f"  Score: {row['suspicion_score']:>5.1f} | ${row['volume']:>12,.0f} | {row['question']}")

Top 10 most suspicious markets:
  Score:  93.6 | $   3,168,666 | Another Iran strike on Israel in 2024?
  Score:  93.4 | $   3,265,663 | Israel military response against Iran in October?
  Score:  81.0 | $   3,538,911 | Will Iran close the Strait of Hormuz in 2025?
  Score:  79.3 | $   4,992,689 | Will the Iranian regime fall in 2025?
  Score:  78.9 | $   2,796,607 | US-Iran nuclear deal in 2025?
  Score:  76.7 | $   1,713,637 | Israel strikes Iran before 2026?
  Score:  76.1 | $   1,725,722 | Will the US officially declare war on Iran in 2025?
  Score:  69.7 | $   1,185,191 | Another US military action against Iran before 2026?
  Score:  69.5 | $   1,317,272 | Iran strike on Israel before December?
  Score:  65.2 | $     852,964 | Will the U.S. invade Iran in 2025?


## Step 5: Merge Polymarket Metrics into Master

Add daily Polymarket volume and suspicion columns to the master timeline. Key event dates receive elevated suspicion; other Iran post days get baseline scores.

In [ ]:
total_iran_volume = pm_markets['volume'].sum()
military_volume = pm_markets[pm_markets['category'] == 'military']['volume'].sum()

# Add Polymarket context to master on key dates
master['polymarket_volume'] = 0.0
master['polymarket_suspicion'] = 0.0

for date_str in key_events:
    mask = master['date'] == pd.Timestamp(date_str)
    if mask.any():
        master.loc[mask, 'polymarket_volume'] = total_iran_volume
        master.loc[mask, 'polymarket_suspicion'] = 80.0

# Add baseline suspicion for other Iran post days
iran_post_mask = (master['iran_posts'] > 0) & (master['polymarket_suspicion'] == 0)
master.loc[iran_post_mask, 'polymarket_volume'] = total_iran_volume * 0.5
master.loc[iran_post_mask, 'polymarket_suspicion'] = 30.0

# Save
master.to_csv(os.path.join(PROCESSED, 'master.csv'), index=False)
suspicion_df.to_csv(os.path.join(PROCESSED, 'polymarket_suspicion.csv'), index=False)

print(f"Updated master.csv with Polymarket columns")
print(f"Saved polymarket_suspicion.csv ({len(suspicion_df)} markets)")

# Print summary stats
print(f"\nPolymarket Summary:")
print(f"  Total Iran market volume: ${total_iran_volume:,.0f}")
print(f"  Military markets volume: ${military_volume:,.0f} ({military_volume/total_iran_volume*100:.1f}%)")
print(f"  Mean suspicion score: {suspicion_df['suspicion_score'].mean():.1f}")
print(f"  Max suspicion score: {suspicion_df['suspicion_score'].max():.1f}")

Updated master.csv with Polymarket columns
Saved polymarket_suspicion.csv (23 markets)

Polymarket Summary:
  Total Iran market volume: $30,048,626
  Military markets volume: $17,107,686 (56.9%)
  Mean suspicion score: 63.5
  Max suspicion score: 93.6
